In [ ]:
# =============================================================================
# Single-cell communication analysis (Seurat + CellChat)
# For human AD dataset, comparing normal vs. senescent cell types
# Place input files in working directory before running
# =============================================================================

library(Seurat)
library(CellChat)
library(Matrix)
library(stringr)
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(ggpubr)

# ----------------------------- Data preparation ------------------------------
# Read annotated Seurat object
seurat_obj <- readRDS("anno_data_0212.rds")

# Subset AD samples
ad_data <- subset(seurat_obj, subset = pheno == "AD")

# Set cell type identity
Idents(ad_data) <- "cell_groups"

# Downsample: max 1000 cells per group (to reduce memory usage)
cat("Original cells:", ncol(ad_data), "\n")
ad_data_subset <- subset(ad_data, downsample = 1000)
cat("After downsampling:", ncol(ad_data_subset), "\n")
print(table(Idents(ad_data_subset)))

# ----------------------------- CellChat analysis -----------------------------
# Extract normalized expression matrix
data.input <- GetAssayData(ad_data_subset, assay = "RNA", slot = "data")
meta.data <- ad_data_subset@meta.data

# Create CellChat object
cellchat <- createCellChat(object = data.input, meta = meta.data, group.by = "cell_groups")

# Load human ligand-receptor database
CellChatDB <- CellChatDB.human
cellchat@DB <- CellChatDB

# Preprocessing
cellchat <- subsetData(cellchat)
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)

# Compute communication probability (core step)
cellchat <- computeCommunProb(cellchat, raw.use = TRUE)

# Filter out interactions involving few cells
cellchat <- filterCommunication(cellchat, min.cells = 10)

# Compute pathway-level communication
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)

cat("CellChat analysis completed!\n")

# ----------------------------- Total communication weight bar plot -----------
# Define cell type order for plotting
cell_order <- c(
  "EXC", "INH", "OLG", "OPC", "ENDO", "FIB", "MAC",
  "Normal AST", "Senescent AST",
  "Normal PER", "Senescent PER",
  "Normal MIC", "Senescent MIC",
  "Normal TC", "Senescent TC"
)

# Define colors (customizable)
my_colors <- setNames(scales::hue_pal()(length(cell_order)), cell_order)

# Calculate total weights (outgoing + incoming)
#     cellchat <- createCellChat(object = obj, group.by = "ident", assay = "RNA")
#     cellchat@DB <- CellChatDB
#     cellchat <- subsetData(cellchat)
#     cellchat <- identifyOverExpressedGenes(cellchat)
#     cellchat <- identifyOverExpressedInteractions(cellchat)
#     cellchat <- computeCommunProb(cellchat, raw.use = TRUE, type = "truncatedMean", trim = 0.1)
#     cellchat <- filterCommunication(cellchat, min.cells = 10)
#     cellchat <- computeCommunProbPathway(cellchat)
#     cellchat <- aggregateNet(cellchat)
#     saveRDS(cellchat, file = file_path)
#   }, error = function(e) message("Error in ", sample_name, ": ", e))
#   rm(obj, cellchat); gc()
# }

# ----------------------------- Read all sample objects -----------------------
# rds_files <- list.files("cellchat_objects_AD", pattern = "_cellchat\\.rds$", full.names = TRUE)
# cellchat_list <- lapply(rds_files, readRDS)
# names(cellchat_list) <- gsub("_cellchat\\.rds$", "", basename(rds_files))

# ----------------------------- Pairwise comparison (Senescent vs Normal) -----
# Define three groups of interesting ligand‑receptor pairs
static_cells <- c("INH", "EXC")   # cell types without condition prefix

get_cell_name <- function(cell_type, condition_prefix) {
  if (cell_type %in% static_cells) return(cell_type)
  else return(paste(condition_prefix, cell_type))
}

group1_pairs <- list(
  list(ligand = "COL4A1", receptor = "CD44", source = "PER", target = "AST"),
  list(ligand = "LAMA2",  receptor = "CD44", source = "PER", target = "AST"),
  list(ligand = "FN1",    receptor = "CD44", source = "PER", target = "AST")
)

group2_pairs <- list(
  list(ligand = "NRG3",   receptor = "ERBB4", source = "INH", target = "PER"),
  list(ligand = "NRG2",   receptor = "ERBB4", source = "AST", target = "INH"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "AST", target = "INH"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "PER", target = "INH"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "MIC", target = "INH"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "AST", target = "EXC"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "PER", target = "EXC"),
  list(ligand = "NRXN1",  receptor = "NLGN1", source = "MIC", target = "EXC")
)

group3_pairs <- list(
  list(ligand = "ANGPTL4", receptor = "CDH11",     source = "AST", target = "PER"),
  list(ligand = "FGF1",    receptor = "FGFR1",     source = "PER", target = "AST"),
  list(ligand = "SPP1",    receptor = "CD44",      source = "MIC", target = "AST"),
  list(ligand = "TGFB1",   receptor = "TGFbR1_R2", source = "MIC", target = "TC")
)

# Example: using a single sample (cellchat) for demonstration
# For multiple samples, replace with a loop over cellchat_list
sample_names <- "Sample1"
cellchat_list <- list(Sample1 = cellchat)

all_results <- data.frame()

for (pair in group1_pairs) {
  src_sen <- get_cell_name(pair$source, "Senescent")
  tar_sen <- get_cell_name(pair$target, "Senescent")
  src_nor <- get_cell_name(pair$source, "Normal")
  tar_nor <- get_cell_name(pair$target, "Normal")

  for (sname in sample_names) {
    df.net <- subsetCommunication(cellchat_list[[sname]])
    if (is.null(df.net)) next

    prob_sen <- df.net %>%
      filter(source == src_sen, target == tar_sen,
             ligand == pair$ligand, receptor == pair$receptor) %>%
      pull(prob)
    prob_sen <- ifelse(length(prob_sen) == 0, 0, prob_sen[1])

    prob_nor <- df.net %>%
      filter(source == src_nor, target == tar_nor,
             ligand == pair$ligand, receptor == pair$receptor) %>%
      pull(prob)
    prob_nor <- ifelse(length(prob_nor) == 0, 0, prob_nor[1])

    all_results <- rbind(all_results, data.frame(
      Sample = sname,
      Ligand = pair$ligand,
      Receptor = pair$receptor,
      Source = pair$source,
      Target = pair$target,
      Senescent = prob_sen,
      Normal = prob_nor
    ))
  }
}

# Generate paired plots
if (nrow(all_results) > 0) {
  results_long <- all_results %>%
    pivot_longer(cols = c("Senescent", "Normal"), names_to = "Condition", values_to = "Probability") %>%
    mutate(Condition = factor(Condition, levels = c("Normal", "Senescent")))

  dir.create("pairwise_comparisons", showWarnings = FALSE)

  for (pair_id in unique(paste(results_long$Ligand, results_long$Receptor, results_long$Source, results_long$Target))) {
    sub <- results_long %>%
      filter(paste(Ligand, Receptor, Source, Target) == pair_id)
    if (nrow(sub) == 0) next

    p <- ggpaired(sub, x = "Condition", y = "Probability",
                  id = "Sample", color = "Condition",
                  palette = c("#00AFBB", "#FC4E07"),
                  line.color = "gray", line.size = 0.4,
                  point.size = 2, xlab = "") +
      stat_compare_means(paired = TRUE, method = "wilcox.test",
                         label = "p.format", label.y.npc = "top") +
      labs(title = paste(unique(sub$Ligand), "-", unique(sub$Receptor)),
           subtitle = paste(unique(sub$Source), "\u2192", unique(sub$Target)),
           y = "Probability") +
      theme(legend.position = "none",
            plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
            plot.subtitle = element_text(size = 9, hjust = 0.5))

    ggsave(file.path("pairwise_comparisons",
                     paste0(unique(sub$Ligand), "-", unique(sub$Receptor), "_",
                            unique(sub$Source), "2", unique(sub$Target), ".pdf")),
           p, width = 6, height = 6)
  }
}

